<a href="https://colab.research.google.com/github/Malaaq-XO/Library-Management-/blob/main/docs/tutorials/qsimcirq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2020 Google

In [1]:
import torch
import torch.nn as nn

# Install pennylane if not already installed
try:
    import pennylane as qml
except ImportError:
    !pip install pennylane --quiet
    import pennylane as qml

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit(inputs, weights):
    # inputs shape: (n_qubits,)
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {"weights": (2, n_qubits, 3)}
        self.layer = qml.qnn.TorchLayer(circuit, weight_shapes)

    def forward(self, x):
        print("\n🔹 Input to Quantum Layer:", x[0].detach().cpu().numpy())

        out = self.layer(x)

        print("⚛️ Output from Quantum Layer:", out[0].detach().cpu().numpy())
        return out


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 93.1 MB/s eta 0:00:00


In [12]:
import torch

# Create sample input data
# The circuit expects inputs of shape (n_qubits,)
# For a batch of inputs, it would be (batch_size, n_qubits)
# Let's create a batch of 2 inputs, each with 4 features (n_qubits)
sample_input = torch.tensor([[0.5, 0.1, 0.9, 0.3], [0.2, 0.8, 0.4, 0.7]], dtype=torch.float32)

print("Sample Input Data:")
print(sample_input)

# Run the sample input data through the QuantumLayer
output = quantum_layer(sample_input)

print("\nOutput from QuantumLayer:")
print(output)

Sample Input Data:
tensor([[0.5000, 0.1000, 0.9000, 0.3000],
        [0.2000, 0.8000, 0.4000, 0.7000]])

🔹 Input to Quantum Layer: [0.5 0.1 0.9 0.3]
⚛️ Output from Quantum Layer: [ 0.12589876  0.02408283 -0.11430131 -0.15681294]

Output from QuantumLayer:
tensor([[ 0.1259,  0.0241, -0.1143, -0.1568],
        [ 0.1927,  0.0735, -0.1287, -0.1484]], grad_fn=<ViewBackward0>)


In [10]:
quantum_layer = QuantumLayer()
print("QuantumLayer instantiated successfully.")

QuantumLayer instantiated successfully.


In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Get started with qsimcirq

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://quantumai.google/qsim/tutorials/qsimcirq"><img src="https://quantumai.google/site-assets/images/buttons/quantumai_logo_1x.png" />View on QuantumAI</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/quantumlib/qsim/blob/main/docs/tutorials/qsimcirq.ipynb"><img src="https://quantumai.google/site-assets/images/buttons/colab_logo_1x.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/quantumlib/qsim/blob/main/docs/tutorials/qsimcirq.ipynb"><img src="https://quantumai.google/site-assets/images/buttons/github_logo_1x.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/qsim/docs/tutorials/qsimcirq.ipynb"><img src="https://quantumai.google/site-assets/images/buttons/download_icon_1x.png" />Download notebook</a>
  </td>
</table>

The qsim library provides a Python interface to Cirq in the **qsimcirq** PyPI package.

## Setup

Install the Cirq and qsimcirq packages: